# Bound entangled states from unextendible product bases (UPBs)

An unextendible product basis is a set of mutually orthogonal product states
whose orthogonal complement contains no product state. The normalised projector
onto that complement, ρ = (I − Π)/(d − |UPB|), is **PPT and entangled** — a bound
entangled state, and therefore a candidate input for the PPT² test.

This notebook builds bound entangled states two ways: from three textbook UPBs
(Shifts, Tiles, Pyramid) with explicit structure, and from UPBs grown
numerically by random restarts. Both are verified PPT, and the random states are
checked one at a time with `test_ppt2`. UPB search in 4⊗4 is slow, so only a few
trials are run here.

In [ ]:
include("common.jl")

## UPB construction

The shared building block turns any UPB into a state. To grow a UPB numerically,
look (with random restarts) for a product vector in the orthogonal complement of
the current set; when none is found the basis is unextendible.

In [ ]:
product_projector(a, b) = (v = kron(a, b); v * v')

function upb_state(upb, dA, dB)
    d = dA * dB
    Π = sum(product_projector(a, b) for (a, b) in upb)
    return Hermitian((I(d) - Π) / (d - length(upb)))
end

function realignment_norm(ρ, dA, dB)
    n = dA * dB
    R = zeros(ComplexF64, n, n)
    for i in 0:dA-1, a in 0:dB-1, j in 0:dA-1, b in 0:dB-1
        R[i*dA+j+1, a*dB+b+1] = ρ[i*dB+a+1, j*dB+b+1]
    end
    return sum(svdvals(R))   # ‖R(ρ)‖₁ > 1 ⇒ entangled
end

orthogonal_to_all(a, b, upb) =
    all(abs(dot(kron(ua, ub), kron(a, b))) < 1e-8 for (ua, ub) in upb)

function find_orthogonal_product_vector(upb, dA, dB; n_trials=5000)
    d = dA * dB
    isempty(upb) && return true, normalize(randn(dA)), normalize(randn(dB))
    V = hcat([normalize(kron(a, b)) for (a, b) in upb]...)
    P_perp = I(d) - V * V'
    for _ in 1:n_trials
        a0 = normalize(randn(dA))
        b0 = normalize(randn(dB))
        v = normalize(P_perp * kron(a0, b0))
        M = reshape(v, dA, dB)
        sv = svdvals(M)
        if sv[1] / sum(sv) > 1 - 1e-6        # nearly Schmidt rank 1
            a_new = normalize(M * M' * a0)
            b_new = normalize(M' * a_new)
            orthogonal_to_all(a_new, b_new, upb) && return true, a_new, b_new
        end
    end
    return false, zeros(dA), zeros(dB)
end

function random_upb(dA, dB; max_size=dA*dB-1, n_trials=5000)
    upb = Tuple{Vector{Float64}, Vector{Float64}}[]
    for _ in 1:max_size
        found, a, b = find_orthogonal_product_vector(upb, dA, dB; n_trials)
        found || break
        push!(upb, (normalize(a), normalize(b)))
    end
    return upb, upb_state(upb, dA, dB)
end

## Known UPBs

Three textbook UPBs give explicit, deterministic bound entangled states — handy
fixed sanity checks alongside the random search:

- **Shifts** and **Tiles** in 3⊗3 — five orthogonal product states each whose
  complement holds no product vector (Bennett et al., PRL 82, 5385 (1999)).
- **Pyramid** in 2⊗4 — five states, the smallest possible UPB (Bravyi, 2004).

Each is built into a state with `upb_state` above.

In [ ]:
e0, e1, e2 = [1.0, 0, 0], [0, 1.0, 0], [0, 0, 1.0]
s3 = (e0 + e1 + e2) / √3

shifts_upb_3x3() = [
    (e0,             (e0 - e1) / √2),
    ((e1 - e2) / √2, e2),
    (e2,             (e1 - e2) / √2),
    ((e0 - e1) / √2, e0),
    (s3,             s3),
]

tiles_upb_3x3() = [
    (e0,             (e0 - e1) / √2),
    (e2,             (e1 - e2) / √2),
    ((e0 - e1) / √2, e2),
    ((e1 - e2) / √2, e0),
    (s3,             s3),
]

function pyramid_upb_2x4()
    θ = [2π * k / 5 for k in 0:4]
    [(normalize([cos(π/4), (-1)^k * sin(π/4)]),
      normalize([1.0, cos(θ[k+1]), sin(θ[k+1]), 0.0])) for k in 0:4]
end

Build each state and confirm it is a valid density operator (smallest eigenvalue
≥ 0) and PPT — every UPB state is bound entangled, so `is_ppt` must hold.

In [ ]:
report(ρ, dA, dB) = (
    ppt = is_ppt(Matrix(ρ), dA, dB),
    min_eig = round(eigmin(Hermitian(Matrix(ρ))), digits=8),
)

Dict(
    "Shifts (3⊗3)"  => report(upb_state(shifts_upb_3x3(), 3, 3), 3, 3),
    "Tiles (3⊗3)"   => report(upb_state(tiles_upb_3x3(), 3, 3), 3, 3),
    "Pyramid (2⊗4)" => report(upb_state(pyramid_upb_2x4(), 2, 4), 2, 4),
)

## Build and verify a random state

Now grow a UPB by random restarts. The resulting 3⊗3 state should be PPT
(`is_ppt` from `ppt2`) and flagged as entangled by the realignment criterion
(‖R(ρ)‖₁ > 1 is sufficient, but it can be inconclusive for bound entangled
states).

In [ ]:
upb, ρ = random_upb(3, 3)
(upb_size = length(upb), ppt = is_ppt(Matrix(ρ), 3, 3), realignment = realignment_norm(Matrix(ρ), 3, 3))

## Feed UPB states into the PPT² test

Sample 4⊗4 UPB states and check each with `test_ppt2` (one candidate per call),
stopping at the first detection.

In [ ]:
forms = load_batches("../pncp_4x4.jld2")
hit = nothing
@showprogress for _ in 1:20
    ρ = Matrix(random_upb(4, 4)[2])
    ev = test_ppt2(ρ; forms=forms)
    if ev !== nothing
        global hit = (state=ρ, evidence=ev)
        break
    end
end
hit